# RAG Scholarship Chatbot - Pure Procedural Version

Get Information about Scholarships provided by Govt. of India
https://scholarships.gov.in/

This notebook implements a complete RAG pipeline using:
- **Pure Procedural Functions** - No classes, just functions
- **Qdrant Cloud** - Remote vector database
- **LangChain & LCEL** - Composable chains
- **Chat History** - Multi-turn conversations

## Step 1: Install Dependencies

In [1]:
! pip install langchain langchain-openai langchain-qdrant qdrant-client gradio pandas datasets python-dotenv -q

## Step 2: Import Libraries

In [2]:
import os
from typing import List, Dict, Any
from dotenv import load_dotenv

# Other imports
from datasets import load_dataset

# Load environment variables
load_dotenv()

print("✓ Base imports successful")

✓ Base imports successful


## Step 3: Global Configuration

All configuration stored in a simple dictionary.

In [3]:
# Configuration dictionary
CONFIG = {
    "openai_api_key": os.getenv("OPENAI_API_KEY"),
    "qdrant_url": os.getenv("QDRANT_URL"),
    "qdrant_api_key": os.getenv("QDRANT_API_KEY"),
    "collection_name": os.getenv("COLLECTION_NAME", "scholarships"),
    "model": os.getenv("LLM_MODEL", "gpt-3.5-turbo"),
    "temperature": float(os.getenv("TEMPERATURE", "0.7")),
    "max_tokens": int(os.getenv("MAX_TOKENS", "500")),
    "retrieval_k": int(os.getenv("RETRIEVAL_K", "3")),
    "chunk_size": int(os.getenv("CHUNK_SIZE", "500")),
    "chunk_overlap": int(os.getenv("CHUNK_OVERLAP", "100")),
    "embedding_model": os.getenv("EMBEDDING_MODEL", "text-embedding-3-small"),
}

# Global state
CHAT_HISTORIES = {}  # Store conversation history per session
RETRIEVER = None  # Will be set during initialization
LLM = None  # Will be set during initialization
CHAINS = {}  # Store built chains

print("✓ Configuration initialized")

✓ Configuration initialized


## Step 4: Configuration Functions

In [4]:
def validate_config() -> None:
    """Validate that all required configuration is present."""
    required_keys = ["openai_api_key", "qdrant_url", "qdrant_api_key"]
    
    for key in required_keys:
        if not CONFIG.get(key):
            raise ValueError(f"{key.upper()} environment variable not set")


def display_config() -> None:
    """Display current configuration."""
    print("\n" + "="*60)
    print("CONFIGURATION")
    print("="*60)
    print(f"OpenAI Model: {CONFIG['model']}")
    print(f"Temperature: {CONFIG['temperature']}")
    print(f"Max Tokens: {CONFIG['max_tokens']}")
    print(f"Retrieval K: {CONFIG['retrieval_k']}")
    print(f"Qdrant URL: {CONFIG['qdrant_url']}")
    print("="*60 + "\n")


# Validate configuration
try:
    validate_config()
    display_config()
except ValueError as e:
    print(f"⚠️  Configuration warning: {e}")
    print("Make sure to set environment variables before initializing system")


CONFIGURATION
OpenAI Model: gpt-3.5-turbo
Temperature: 0.7
Max Tokens: 500
Retrieval K: 3
Qdrant URL: https://65a0c75c-b421-4fbc-b600-35181e279381.eu-central-1-0.aws.cloud.qdrant.io



## Step 5: Data Loading Functions

In [5]:
def load_dataset_from_huggingface() -> List[Dict[str, Any]]:
    """Load scholarship dataset from Hugging Face."""
    print("\nLoading scholarship dataset from Hugging Face...")
    dataset = load_dataset("NetraVerse/indian-govt-scholarships")
    df = dataset["train"].to_pandas()
    df = df[['label', 'text']]
    data = df.to_dict('records')
    print(f"✓ Loaded {len(data)} scholarship documents")
    return data


def display_sample_data(data: List[Dict[str, Any]]) -> None:
    """Display sample data from the dataset."""
    if data:
        print(f"\nFirst document example:")
        print(f"Label: {data[0]['label']}")
        print(f"Text (first 200 chars): {data[0]['text'][:200]}...")


# Load data
print("Loading data...")
data = load_dataset_from_huggingface()
display_sample_data(data)

Loading data...

Loading scholarship dataset from Hugging Face...
✓ Loaded 10 scholarship documents

First document example:
Label: AICTE_2010_F
Text (first 200 chars): Page 1 of 8 
  
 
PRAGATI SCHOLARSHIP SCHEME  
Frequently Asked Questions (FAQs)  
Q.1 Who is eligible for PRAGATI Scholarship?  
Ans: Eligibility criteria underPRAGATI Scholarship scheme:  
Eligibili...


## Step 6: Text Processing Functions

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def create_text_splitter():
    """Create a text splitter for chunking documents."""
    return RecursiveCharacterTextSplitter(
        chunk_size=CONFIG["chunk_size"],
        chunk_overlap=CONFIG["chunk_overlap"],
        separators=["\n\n", "\n", " ", ""]
    )

def process_documents(data: List[Dict[str, Any]]) -> List[Document]:
    text_splitter = create_text_splitter()
    texts = [d["text"] for d in data]
    metadatas = [{"source": d["label"], "scholarship_name": d["label"]} for d in data]
    documents = text_splitter.create_documents(texts=texts, metadatas=metadatas)
    print(f"✓ Created {len(documents)} document chunks")
    return documents


# Process documents
documents = process_documents(data)

✓ Created 457 document chunks


## Step 7: Vector Store Setup

In [7]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

def create_embeddings():
    """Create embeddings model."""
    print("\nInitializing embeddings...")
    embeddings = OpenAIEmbeddings(
        model=CONFIG["embedding_model"],
        api_key=CONFIG["openai_api_key"]
    )
    print("✓ Embeddings initialized")
    return embeddings


def create_vector_store(documents: List[Document], embeddings):
    """Create vector store in Qdrant Cloud using QdrantVectorStore."""
    print("\nCreating vector store in Qdrant Cloud...")
    # Extract texts and metadatas from documents
    texts = [doc.page_content for doc in documents]
    metadatas = [doc.metadata for doc in documents]

    # Create vector store using QdrantVectorStore
    vector_store = QdrantVectorStore.from_texts(
        texts=texts,
        embedding=embeddings,
        metadatas=metadatas,
        url=CONFIG["qdrant_url"],
        api_key=CONFIG["qdrant_api_key"],
        collection_name=CONFIG["collection_name"]
    )
    print(f"✓ Vector store created in cloud collection: {CONFIG['collection_name']}")
    return vector_store


def create_retriever(vector_store):
    """Create retriever from vector store."""
    retriever = vector_store.as_retriever(
        search_kwargs={"k": CONFIG["retrieval_k"]}
    )
    print(f"✓ Retriever created (k={CONFIG['retrieval_k']})")
    return retriever


# Create embeddings and vector store
embeddings = create_embeddings()
vector_store = create_vector_store(documents, embeddings)
RETRIEVER = create_retriever(vector_store)


Initializing embeddings...
✓ Embeddings initialized

Creating vector store in Qdrant Cloud...
✓ Vector store created in cloud collection: scholarships
✓ Retriever created (k=3)


## Step 8: Initialize LLM

In [8]:
from langchain_openai import ChatOpenAI

print("\nInitializing LLM...")
LLM = ChatOpenAI(
    model=CONFIG["model"],
    temperature=CONFIG["temperature"],
    max_tokens=CONFIG["max_tokens"],
    api_key=CONFIG["openai_api_key"]
)
print("✓ LLM initialized")


Initializing LLM...
✓ LLM initialized


## Step 9: Document Formatting Functions

In [9]:
def format_documents_for_context(documents: List[Document]) -> str:
    """Format retrieved documents into context string."""
    if not documents:
        return "No documents found."
    
    formatted_parts = []
    for doc in documents:
        scholarship_name = doc.metadata.get('scholarship_name', 'Unknown')
        formatted_parts.append(
            f"**{scholarship_name}**\n{doc.page_content}"
        )
    
    return "\n\n".join(formatted_parts)


def extract_sources(documents: List[Document]) -> str:
    """Extract unique source scholarships from documents."""
    sources = set()
    for doc in documents:
        source = doc.metadata.get('scholarship_name', 'Unknown')
        sources.add(source)
    
    if sources:
        return " | ".join(sorted(list(sources)))
    return "No sources found"


print("✓ Formatting functions created")

✓ Formatting functions created


## Step 10: Query Analysis Functions

In [10]:
def normalize_query(query: str) -> str:
    """Normalize user query text."""
    return query.strip()


def prepare_query_input(query: str) -> Dict[str, Any]:
    """Prepare standardized query input dictionary."""
    return {
        "query": normalize_query(query)
    }


print("✓ Query preparation functions created")

✓ Query preparation functions created


## Step 11: Prompt and Chain Building

In [11]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

SYSTEM_PROMPT = """You are a helpful scholarship advisor.
Answer the user query clearly and directly using the provided context.

Context:
{context}"""


def create_prompt():
    return ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "{query}"),
    ])


def assign_context_to_input(input_dict: Dict[str, Any]) -> Dict[str, Any]:
    """Assign context from retriever to input dictionary."""
    query = input_dict.get("query", "")
    documents = RETRIEVER.invoke(query)
    context = format_documents_for_context(documents)
    input_dict["context"] = context
    return input_dict


def build_main_chain():
    prompt = create_prompt()
    chain = (
        RunnableLambda(assign_context_to_input)
        | prompt
        | LLM
        | StrOutputParser()
    )
    return chain


def build_all_chains():
    """Build chain and store it."""
    print("\nBuilding LCEL chain...")
    CHAINS["main"] = build_main_chain()
    print("✓ Chain built successfully")


# Build chain
build_all_chains()


Building LCEL chain...
✓ Chain built successfully


## Step 12: Routing Functions

In [12]:
def run_chain(input_dict: Dict[str, Any]) -> str:
    """Run the single main LangChain chain."""
    return CHAINS["main"].invoke(input_dict)




## Step 13: Chat History Functions

In [13]:
def get_or_create_history(session_id: str) -> List:
    """Get or create chat history for a session."""
    if session_id not in CHAT_HISTORIES:
        CHAT_HISTORIES[session_id] = []
    return CHAT_HISTORIES[session_id]


def add_message_to_history(session_id: str, message) -> None:
    """Add message to chat history."""
    history = get_or_create_history(session_id)
    history.append(message)


def clear_session(session_id: str) -> None:
    """Clear chat history for a session."""
    if session_id in CHAT_HISTORIES:
        CHAT_HISTORIES[session_id].clear()


def get_session_info(session_id: str) -> Dict[str, Any]:
    """Get information about a session."""
    history = get_or_create_history(session_id)
    return {
        "session_id": session_id,
        "message_count": len(history),
        "turns": len(history) // 2
    }


print("✓ Chat history functions created")

✓ Chat history functions created


## Step 14: Main Chatbot Function

In [14]:
from langchain_core.messages import HumanMessage, AIMessage

def process_query(user_message: str, session_id: str = "default") -> Dict[str, Any]:
    """
    Process a user query and generate response.

    Args:
        user_message: User's input message
        session_id: Session ID for tracking conversation history

    Returns:
        Dictionary with response, sources, and metadata
    """

    # Get chat history
    history = get_or_create_history(session_id)

    # Add user message to history
    add_message_to_history(session_id, HumanMessage(content=user_message))

    # Prepare input and run chain
    input_dict = prepare_query_input(user_message)
    input_dict["chat_history"] = history
    response = run_chain(input_dict)

    # Get sources
    documents = RETRIEVER.invoke(user_message)
    sources = extract_sources(documents)

    # Add assistant response to history
    add_message_to_history(session_id, AIMessage(content=response))

    # Format final response
    final_response = f"{response}\n\n📚 **Sources:** {sources}"

    return {
        "response": response,
        "sources": sources,
        "full_response": final_response,
        "conversation_turns": len(history) // 2,
        "session_id": session_id
    }


print("✓ Main chatbot function created")

✓ Main chatbot function created


## Step 15: Test the Chatbot

In [15]:
print("\n" + "="*80)
print("TESTING THE CHATBOT")
print("="*80 + "\n")

test_queries = [
    "What scholarships are available for engineering students?",
    "Tell me about AICTE scholarships",
    "How do I apply for government scholarships?",
]

session_id = "test_session"

for i, query in enumerate(test_queries, 1):
    print(f"\nTest {i}:")
    print(f"👤 User: {query}")
    
    result = process_query(query, session_id)
    
    print(f"\n🤖 Assistant: {result['response'][:300]}...")
    print(f"\n📚 Sources: {result['sources']}")
    print(f"\n{'='*80}")

# Display session info
session_info = get_session_info(session_id)
print(f"\nSession Info: {session_info}")


TESTING THE CHATBOT


Test 1:
👤 User: What scholarships are available for engineering students?

🤖 Assistant: For engineering students, the scholarship available under the Central Sector Scheme of Scholarship for College and University Students (CSSS) is as follows:
- Rs. 12,000 per annum for the first three years of B.Tech/BE courses
- Rs. 20,000 per annum for the fourth year of B.Tech/BE courses

This sch...

📚 Sources: FAQ_DOHE_CSSS


Test 2:
👤 User: Tell me about AICTE scholarships

🤖 Assistant: AICTE offers scholarships of Rs. 50,000/- per annum for every year of study to students from specific states and union territories. This scholarship is available for a maximum of 4 years for first-year admitted students and a maximum of 3 years for second-year admitted students. The eligible states ...

📚 Sources: AICTE_2010_G


Test 3:
👤 User: How do I apply for government scholarships?

🤖 Assistant: To apply for government scholarships, you can follow these steps:

1. Visit http://www.sc

## Step 16: Test Multi-turn Conversation

Test that the chatbot remembers context across multiple turns.

In [16]:
print("\n" + "="*80)
print("TESTING MULTI-TURN CONVERSATION")
print("="*80 + "\n")

user_session = "user_conversation"

# Turn 1
print("Turn 1:")
result1 = process_query("What scholarships for women?", user_session)
print(f"User: What scholarships for women?")
print(f"Assistant: {result1['response'][:200]}...")

# Turn 2 - should have context from Turn 1
print("\nTurn 2:")
result2 = process_query("Tell me more about STEM scholarships", user_session)
print(f"User: Tell me more about STEM scholarships")
print(f"Assistant: {result2['response'][:200]}...")

# Check history
info = get_session_info(user_session)
print(f"\nConversation Info: {info}")


TESTING MULTI-TURN CONVERSATION

Turn 1:
User: What scholarships for women?
Assistant: The scholarship mentioned is specifically for girls pursuing technical education to empower women with knowledge, skills, and self-confidence. It aims to provide young women with opportunities to furt...

Turn 2:
User: Tell me more about STEM scholarships
Assistant: The scholarship mentioned in the context is specifically for girls pursuing technical education as part of empowering women through technical education. This scholarship aims to provide young women wi...

Conversation Info: {'session_id': 'user_conversation', 'message_count': 4, 'turns': 2}


## Summary

✅ **Loaded data** from Hugging Face
✅ **Created embeddings** using OpenAI
✅ **Connected to Qdrant Cloud** vector database
✅ **Built LCEL chains**
✅ **Added chat history** for multi-turn conversations
✅ **Created main chatbot function** for easy usage
✅ **Tested everything** with sample queries

**All implemented with pure procedural functions - no classes!**